In [1]:
import pandas as pd
import time
# Vì content_model.py nằm chung thư mục models với file này
from content_model import TFIDFRecommender, SBERTExtractor

print("="*60)
print("🚀 BẮT ĐẦU QUÁ TRÌNH TRÍCH XUẤT ĐẶC TRƯNG VĂN BẢN (SBERT)")
print("="*60)

# Bắt đầu bấm giờ
start_time = time.time()

print("\n[Bước 1/3] Đang đọc dữ liệu từ thư mục cấp cha '../data/sample/'...")
movies_df = pd.read_csv('../data/sample/movies.csv')
enriched_df = pd.read_csv('../data/sample/enriched_movies.csv')
print(" -> Đọc các file CSV thành công!")

print("\n[Bước 2/3] Đang gộp dữ liệu phim (Merge)...")
df = pd.merge(movies_df, enriched_df[['movieId', 'overview']], on='movieId', how='left')
df = df.reset_index(drop=True)
print(f" -> ✅ Đã chuẩn bị xong dữ liệu cho {len(df)} bộ phim.")

print("\n[Bước 3/3] Khởi động AI và trích xuất Vector (Vui lòng chờ)...")
print("-" * 60)
sbert_model = SBERTExtractor()

# CẬP NHẬT TẠI ĐÂY: Truyền thêm đường dẫn '../artifacts' để lưu ra folder ngoài cùng
sbert_model.extract_and_save(df, artifact_path='../artifacts') 
print("-" * 60)

# Dừng bấm giờ và tính toán
end_time = time.time()
execution_time = (end_time - start_time) / 60

print(f"\n🎉 HOÀN TẤT TOÀN BỘ QUÁ TRÌNH!")
print(f"⏱️ Tổng thời gian chạy: {execution_time:.2f} phút.")
print("="*60)

c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 BẮT ĐẦU QUÁ TRÌNH TRÍCH XUẤT ĐẶC TRƯNG VĂN BẢN (SBERT)

[Bước 1/3] Đang đọc dữ liệu từ thư mục cấp cha '../data/sample/'...
 -> Đọc các file CSV thành công!

[Bước 2/3] Đang gộp dữ liệu phim (Merge)...
 -> ✅ Đã chuẩn bị xong dữ liệu cho 25 bộ phim.

[Bước 3/3] Khởi động AI và trích xuất Vector (Vui lòng chờ)...
------------------------------------------------------------
[SBERT] Đang tải mạng nơ-ron ngôn ngữ all-mpnet-base-v2...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6074.85it/s]


[SBERT] Đang chuẩn bị văn bản ngữ nghĩa...
[SBERT] Bắt đầu mã hóa (Sẽ tốn thời gian, vui lòng chờ)...


Batches: 100%|██████████| 4/4 [00:01<00:00,  3.29it/s]

[SBERT] Hoàn tất! Đã lưu vector 768-chiều tại ../artifacts/movie_embeddings.npy
------------------------------------------------------------

🎉 HOÀN TẤT TOÀN BỘ QUÁ TRÌNH!
⏱️ Tổng thời gian chạy: 0.12 phút.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
import sys
import os

# 1. Báo cho Python biết cách tìm thư mục 'data' và 'models'
sys.path.append(os.path.abspath('..'))

# 2. Import "Băng chuyền" (Dataloader) và "Bộ não" (Mô hình)
from data.dataloader2 import get_twotower_dataloaders
from models.TwoTower2 import TwoTowerModel

print("="*60)
print("🚀 KHỞI ĐỘNG KỊCH BẢN HUẤN LUYỆN THÁP ĐÔI (BPR LOSS)")
print("="*60)

# ==========================================
# CẤU HÌNH THÔNG SỐ (HYPERPARAMETERS)
# ==========================================
EPOCHS = 10               # Số vòng lặp qua toàn bộ dữ liệu
BATCH_SIZE = 64           # Số lượng cặp Âm-Dương học cùng lúc
LEARNING_RATE = 0.001     # Tốc độ học (Cẩn thận: Lớn quá sẽ "ngu", nhỏ quá sẽ chậm)
EMBEDDING_DIM = 768       # Kích thước vector chuẩn của SBERT

# Sử dụng Card Đồ Họa (GPU) nếu có, nếu không thì xài CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Đang sử dụng thiết bị tính toán: {device}")

# ==========================================
# CHUẨN BỊ BĂNG CHUYỀN VÀ MÔ HÌNH
# ==========================================
# Gọi hàm mà chúng ta đã chuẩn bị ở Bước 2
# Nhớ trỏ đường dẫn đúng vào thư mục chứa dữ liệu thô và SBERT artifacts
train_loader, val_df, test_df = get_twotower_dataloaders(
    data_dir='../data/sample', 
    artifact_dir='../artifacts', 
    batch_size=BATCH_SIZE
)

# Khởi tạo mạng nơ-ron từ Bước 3 và đẩy lên thiết bị
model = TwoTowerModel(input_dim=EMBEDDING_DIM).to(device)

# Thuật toán tối ưu hóa (Optimizer) Adam - Giúp mô hình cập nhật trọng số thông minh
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ==========================================
# HÀM MẤT MÁT (BPR LOSS) - TRÁI TIM CỦA GỢI Ý
# ==========================================
def bpr_loss(pos_scores, neg_scores):
    """
    Bayesian Personalized Ranking (BPR) Loss:
    Mục tiêu: Làm cho điểm số của Phim Dương (pos) luôn lớn hơn Phim Âm (neg).
    Công thức: -log(sigmoid(pos_score - neg_score))
    """
    # Tính khoảng cách biệt giữa phim thích và không thích
    distances = pos_scores - neg_scores
    # Dùng hàm Sigmoid ép khoảng cách đó vào [0, 1] rồi tính Log
    loss = -torch.mean(torch.nn.functional.logsigmoid(distances))
    return loss

# ==========================================
# VÒNG LẶP HUẤN LUYỆN (TRAINING LOOP)
# ==========================================
print("\n🔥 BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN...")
start_time = time.time()

# Đặt mô hình ở chế độ Huấn luyện (Kích hoạt Dropout)
model.train()

for epoch in range(EPOCHS):
    total_loss = 0.0
    
    # Lấy từng lô dữ liệu từ Dataloader
    for batch_idx, (user_vec, pos_vec, neg_vec) in enumerate(train_loader):
        # Đẩy dữ liệu lên GPU/CPU
        user_vec = user_vec.to(device)
        pos_vec = pos_vec.to(device)
        neg_vec = neg_vec.to(device)
        
        # 1. Reset gradient (Xóa rác của bước trước)
        optimizer.zero_grad()
        
        # 2. Lan truyền tiến (Forward Pass)
        # Bắt mô hình dự đoán điểm số cho cặp Dương và cặp Âm
        pos_scores = model(user_vec, pos_vec)
        neg_scores = model(user_vec, neg_vec)
        
        # 3. Tính toán hình phạt (Loss) nếu dự đoán sai
        loss = bpr_loss(pos_scores, neg_scores)
        
        # 4. Lan truyền ngược (Backward Pass) để tìm ra cách sửa lỗi
        loss.backward()
        
        # 5. Cập nhật trọng số của mạng nơ-ron
        optimizer.step()
        
        total_loss += loss.item()
    
    # In ra báo cáo tiến độ sau mỗi Epoch
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{EPOCHS}] | BPR Loss (Hình phạt): {avg_loss:.4f}")

# Đo thời gian kết thúc
end_time = time.time()
print(f"\n🎉 HUẤN LUYỆN THÀNH CÔNG! Thời gian chạy: {(end_time - start_time)/60:.2f} phút.")

# ==========================================
# BÀN GIAO SẢN PHẨM CUỐI CÙNG
# ==========================================
# Lưu trọng số mô hình ra ổ cứng để backend có thể mang đi phục vụ User
torch.save(model.state_dict(), '../artifacts/twotower_model.pth')
print("💾 Đã lưu trọng số mạng nơ-ron tại: '../artifacts/twotower_model.pth'")
print("="*60)

🚀 KHỞI ĐỘNG KỊCH BẢN HUẤN LUYỆN THÁP ĐÔI (BPR LOSS)
🖥️ Đang sử dụng thiết bị tính toán: cpu
[DataLoader2] Tiến hành kích hoạt bộ phân rã chia tách Train/Val/Test theo dòng thời gian...
[Dataset] Đang nạp ma trận nhúng SBERT từ: ../artifacts/movie_embeddings.npy
[Dataset] Đang pre-compute User Profile (Tính trung bình vector đặc trưng)...
[Dataset] Đã tạo xong 42 cặp tương tác tích cực (Mẫu Dương).

🔥 BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN...
Epoch [1/10] | BPR Loss (Hình phạt): 0.6926
Epoch [2/10] | BPR Loss (Hình phạt): 0.6882
Epoch [3/10] | BPR Loss (Hình phạt): 0.6787
Epoch [4/10] | BPR Loss (Hình phạt): 0.6509
Epoch [5/10] | BPR Loss (Hình phạt): 0.6520
Epoch [6/10] | BPR Loss (Hình phạt): 0.6021
Epoch [7/10] | BPR Loss (Hình phạt): 0.6078
Epoch [8/10] | BPR Loss (Hình phạt): 0.5664
Epoch [9/10] | BPR Loss (Hình phạt): 0.6052
Epoch [10/10] | BPR Loss (Hình phạt): 0.5378

🎉 HUẤN LUYỆN THÀNH CÔNG! Thời gian chạy: 0.01 phút.
💾 Đã lưu trọng số mạng nơ-ron tại: '../artifacts/twotower_model.pth'


In [3]:
import torch
import numpy as np
import pandas as pd

print("="*60)
print("🍿 CHẠY THỬ NGHIỆM HỆ THỐNG GỢI Ý THÁP ĐÔI")
print("="*60)

# 1. Bật chế độ suy luận (Tắt các lớp Dropout để dự đoán ổn định)
model.eval()

# 2. Chuẩn bị dữ liệu để dự đoán
USER_ID = 101  # Bạn có thể đổi sang 102, 103... để xem gu mỗi người khác nhau thế nào
print(f"Đang phân tích gu điện ảnh của User {USER_ID}...")

# Trích xuất User Profile đã được lưu trong Dataset
# (Lấy từ train_loader mà bạn đã chạy ở cell trên)
user_dataset = train_loader.dataset
if USER_ID not in user_dataset.user_profiles:
    print(f"❌ Không tìm thấy lịch sử của User {USER_ID}!")
else:
    # Lấy vector 768-chiều của User
    user_vec = user_dataset.user_profiles[USER_ID]
    user_tensor = torch.tensor(user_vec, dtype=torch.float32).unsqueeze(0).to(device)
    
    # Nạp toàn bộ vector của TẤT CẢ các bộ phim trong kho (Để chấm điểm thi)
    all_movies_tensor = torch.tensor(user_dataset.embeddings, dtype=torch.float32).to(device)
    
    # 3. Yêu cầu Tháp Đôi chấm điểm toàn bộ phim
    with torch.no_grad(): # Tắt tính đạo hàm (giúp chạy cực nhanh)
        # Nhờ tính chất Broadcast của PyTorch, ta có thể chấm điểm 1 User với hàng ngàn Phim cùng lúc!
        scores = model(user_tensor, all_movies_tensor)
        
    # 4. Lọc ra Top 5 phim có điểm số cao nhất
    top_k = 5
    top_scores, top_indices = torch.topk(scores, top_k)
    
    # Map ngược từ Index sang Movie ID và Tên phim
    reverse_map = {idx: movie_id for movie_id, idx in user_dataset.movie_map.items()}
    movies_df = pd.read_csv('../data/sample/movies.csv')
    
    print(f"\n🎬 TOP {top_k} BỘ PHIM AI KHUYÊN USER {USER_ID} NÊN XEM:")
    print("-" * 50)
    for i in range(top_k):
        idx = top_indices[i].item()
        score = top_scores[i].item()
        
        real_movie_id = reverse_map[idx]
        movie_title = movies_df[movies_df['movieId'] == real_movie_id]['title'].values[0]
        movie_genres = movies_df[movies_df['movieId'] == real_movie_id]['genres'].values[0]
        
        print(f"⭐ Hạng {i+1}: {movie_title}")
        print(f"   - Thể loại: {movie_genres}")
        print(f"   - Độ khớp gu: {score:.4f}\n")

🍿 CHẠY THỬ NGHIỆM HỆ THỐNG GỢI Ý THÁP ĐÔI
Đang phân tích gu điện ảnh của User 101...

🎬 TOP 5 BỘ PHIM AI KHUYÊN USER 101 NÊN XEM:
--------------------------------------------------
⭐ Hạng 1: Back to the Future (1985)
   - Thể loại: Adventure|Comedy|Sci-Fi
   - Độ khớp gu: 0.7537

⭐ Hạng 2: Babe (1995)
   - Thể loại: Children|Drama
   - Độ khớp gu: 0.7414

⭐ Hạng 3: American President The (1995)
   - Thể loại: Comedy|Drama|Romance
   - Độ khớp gu: 0.7401

⭐ Hạng 4: Braveheart (1995)
   - Thể loại: Action|Drama|War
   - Độ khớp gu: 0.7253

⭐ Hạng 5: Taxi Driver (1976)
   - Thể loại: Crime|Drama|Thriller
   - Độ khớp gu: 0.6914



Test SBERT

In [1]:
import pandas as pd
import sys
import os

# 1. Báo cho Python biết thư mục gốc của dự án nằm ở đâu
sys.path.append(os.path.abspath('..'))

# 2. Import class SBERTRecommender từ file mã nguồn của bạn
from src.models.content_model import SBERTRecommender

if __name__ == "__main__":
    print("--- KHỞI TẠO DỮ LIỆU GIẢ LẬP ---")
    movies_data = pd.DataFrame({
        'movie_id': [1, 2, 3],
        'title': ['Toy Story', 'Jumanji', 'Grumpier Old Men'],
        'genres': ['Adventure|Animation|Children|Comedy|Fantasy', 'Adventure|Children|Fantasy', 'Comedy|Romance'],
        'tags': ['pixar toys', 'board game magic', 'old men'],
        'overview': ['A cowboy doll is profoundly threatened...', 'When two kids find and play a magical board game...', 'A family wedding reignites the ancient feud...']
    })
    
    user_history_data = pd.DataFrame({
        'user_id': [99, 99],
        'movie_id': [1, 2] # User 99 đã xem Toy Story và Jumanji
    })

    # 3. Khởi tạo và Huấn luyện bằng SBERT
    print("\n===========================================")
    print("        BẮT ĐẦU CHẠY MÔ HÌNH SBERT         ")
    print("===========================================")
    sbert_model = SBERTRecommender()
    sbert_model.fit(movies_data)

    # 4. Test các hàm chức năng
    print("\n--- TEST: Phim giống Toy Story (SBERT) ---")
    print(sbert_model.recommend_similar_movies(movie_id=1, top_k=1))

    print("\n--- TEST: Phim gợi ý cho User 99 (SBERT) ---")
    print(sbert_model.recommend_content_for_user(user_id=99, user_history_df=user_history_data, top_k=1))

c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- KHỞI TẠO DỮ LIỆU GIẢ LẬP ---

        BẮT ĐẦU CHẠY MÔ HÌNH SBERT         
[SBERT] Đang tải mô hình bản đầy đủ (all-mpnet-base-v2)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3168.89it/s]


[SBERT] Đang mã hóa văn bản (batch_size=8 để bảo vệ RAM 8GB)...


Batches: 100%|██████████| 1/1 [00:03<00:00,  3.05s/it]

[SBERT] Đang tính toán Cosine Similarity...
[SBERT] Đã lưu ma trận nhúng (768 chiều) tại ../artifacts/movie_embeddings.npy
[SBERT] Đã lưu ma trận tương đồng tại ../artifacts/content_similarity.npy

--- TEST: Phim giống Toy Story (SBERT) ---
   movie_id    title                      genres
1         2  Jumanji  Adventure|Children|Fantasy

--- TEST: Phim gợi ý cho User 99 (SBERT) ---
   movie_id             title          genres
2         3  Grumpier Old Men  Comedy|Romance


Test TFIDF

In [1]:
import pandas as pd
import sys
import os

# 1. Báo cho Python biết thư mục gốc của dự án nằm ở đâu (lùi lại 1 cấp từ thư mục notebooks)
sys.path.append(os.path.abspath('..'))

# 2. Import trực tiếp class TFIDFRecommender từ file mã nguồn của bạn
from src.models.content_model import TFIDFRecommender

if __name__ == "__main__":
    print("--- KHỞI TẠO DỮ LIỆU GIẢ LẬP ---")
    movies_data = pd.DataFrame({
        'movie_id': [1, 2, 3],
        'title': ['Toy Story', 'Jumanji', 'Grumpier Old Men'],
        'genres': ['Adventure|Animation|Children|Comedy|Fantasy', 'Adventure|Children|Fantasy', 'Comedy|Romance'],
        'tags': ['pixar toys', 'board game magic', 'old men'],
        'overview': ['A cowboy doll is profoundly threatened...', 'When two kids find and play a magical board game...', 'A family wedding reignites the ancient feud...']
    })
    
    user_history_data = pd.DataFrame({
        'user_id': [99, 99],
        'movie_id': [1, 2] # User 99 đã xem Toy Story và Jumanji
    })

    # 3. Khởi tạo và Huấn luyện bằng class đã import
    recommender = TFIDFRecommender()
    recommender.fit(movies_data)

    # 4. Test các hàm chức năng
    print("\n--- TEST: Phim giống Toy Story ---")
    print(recommender.recommend_similar_movies(movie_id=1, top_k=1))

    print("\n--- TEST: Phim gợi ý cho User 99 ---")
    print(recommender.recommend_content_for_user(user_id=99, user_history_df=user_history_data, top_k=1))

c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- KHỞI TẠO DỮ LIỆU GIẢ LẬP ---
[TF-IDF] Đang tính toán ma trận TF-IDF...
[TF-IDF] Đang tính toán Cosine Similarity...
[TF-IDF] Đã lưu mô hình dự phòng tại ../artifacts/tfidf_similarity.npy

--- TEST: Phim giống Toy Story ---
   movie_id    title                      genres
1         2  Jumanji  Adventure|Children|Fantasy

--- TEST: Phim gợi ý cho User 99 ---
   movie_id             title          genres
2         3  Grumpier Old Men  Comedy|Romance


Test máy có chạy được sbert không

In [7]:
# Cài đặt thư viện kiểm tra (chạy lệnh này trong Terminal nếu máy báo thiếu module)
# pip install torch psutil

import torch
import psutil

print("=== KẾT QUẢ KIỂM TRA PHẦN CỨNG CHO SBERT ===")

# 1. Kiểm tra RAM hệ thống
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"1. RAM Hệ thống: {ram_gb:.2f} GB")
if ram_gb < 8:
    print("   -> Cảnh báo: RAM hơi yếu, có thể bị tràn bộ nhớ nếu dữ liệu quá lớn.")
else:
    print("   -> Đạt yêu cầu chạy SBERT.")

# 2. Kiểm tra Card đồ họa (GPU)
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"\n2. Card đồ họa (GPU): TÌM THẤY")
    print(f"   -> Tên Card: {device_name}")
    print(f"   -> VRAM: {vram_gb:.2f} GB")
    print("\n=> KẾT LUẬN: Tuyệt vời! Máy bạn có thể huấn luyện SBERT siêu tốc bằng GPU.")
else:
    print(f"\n2. Card đồ họa (GPU): KHÔNG TÌM THẤY (Hoặc chưa cài CUDA)")
    print("\n=> KẾT LUẬN: Máy bạn sẽ chạy SBERT bằng CPU. Quá trình này sẽ mất thời gian hơn một chút, chúng ta sẽ cần chọn mô hình SBERT loại nhỏ (MiniLM) để tối ưu tốc độ.")

=== KẾT QUẢ KIỂM TRA PHẦN CỨNG CHO SBERT ===
1. RAM Hệ thống: 7.85 GB
   -> Cảnh báo: RAM hơi yếu, có thể bị tràn bộ nhớ nếu dữ liệu quá lớn.

2. Card đồ họa (GPU): KHÔNG TÌM THẤY (Hoặc chưa cài CUDA)

=> KẾT LUẬN: Máy bạn sẽ chạy SBERT bằng CPU. Quá trình này sẽ mất thời gian hơn một chút, chúng ta sẽ cần chọn mô hình SBERT loại nhỏ (MiniLM) để tối ưu tốc độ.
